<a href="https://colab.research.google.com/github/hrishykesh/Hundimba/blob/main/SeqReversalTransformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:

import torch
import torch.nn as nn
import torch.nn.functional as F

import numpy as np
import random
import math

from torch.utils.data import TensorDataset, DataLoader

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# Select computing device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch version:", torch.__version__)
print("Computing device:", device)


PyTorch version: 2.14.1+cpu
Computing device: cpu


In [9]:

# Our character vocabulary
characters = list("abcdef")

# Special token for beginning of decoder input
BOS = "<BOS>"

vocab = characters + [BOS]

# Character -> Integer
stoi = {ch: i for i, ch in enumerate(vocab)}

# Integer -> Character
itos = {i: ch for ch, i in stoi.items()}

vocab_size = len(vocab)

print("Vocabulary:", vocab)
print("Vocabulary size:", vocab_size)
print("Character mapping:", stoi)


Vocabulary: ['a', 'b', 'c', 'd', 'e', 'f', '<BOS>']
Vocabulary size: 7
Character mapping: {'a': 0, 'b': 1, 'c': 2, 'd': 3, 'e': 4, 'f': 5, '<BOS>': 6}


In [10]:

def encode(text):
    return [stoi[ch] for ch in text]


def decode(tokens):
    return "".join(
        itos[int(token)]
        for token in tokens
        if int(token) != stoi[BOS]
    )


# Test
sample = "abcdef"

encoded = encode(sample)

print("Original:", sample)
print("Encoded:", encoded)
print("Decoded:", decode(encoded))


Original: abcdef
Encoded: [0, 1, 2, 3, 4, 5]
Decoded: abcdef


In [11]:

def generate_sample(seq_length=8):

    # Generate random characters
    source = [
        random.choice(characters)
        for _ in range(seq_length)
    ]

    # Reverse the sequence
    target = source[::-1]

    # Convert characters to integer IDs
    source_ids = encode(source)

    target_ids = encode(target)

    # Shift target right by one position
    decoder_input = [stoi[BOS]] + target_ids[:-1]

    return source_ids, decoder_input, target_ids


# Inspect one example
src, dec_in, tgt = generate_sample()

print("Source:", decode(src))
print("Decoder input:", dec_in)
print("Expected output:", decode(tgt))


Source: faafcbbb
Decoder input: [6, 1, 1, 1, 2, 5, 0, 0]
Expected output: bbbcfaaf


In [12]:

def create_dataset(num_samples, seq_length=8):

    sources = []
    decoder_inputs = []
    targets = []

    for _ in range(num_samples):

        src, dec_in, tgt = generate_sample(seq_length)

        sources.append(src)
        decoder_inputs.append(dec_in)
        targets.append(tgt)

    return (
        torch.tensor(sources, dtype=torch.long),
        torch.tensor(decoder_inputs, dtype=torch.long),
        torch.tensor(targets, dtype=torch.long)
    )


# Generate datasets
X_train, D_train, Y_train = create_dataset(12000)
X_val, D_val, Y_val = create_dataset(2000)

print("Training source shape:", X_train.shape)
print("Training decoder shape:", D_train.shape)
print("Training target shape:", Y_train.shape)


Training source shape: torch.Size([12000, 8])
Training decoder shape: torch.Size([12000, 8])
Training target shape: torch.Size([12000, 8])


In [13]:

batch_size = 128

train_dataset = TensorDataset(
    X_train,
    D_train,
    Y_train
)

val_dataset = TensorDataset(
    X_val,
    D_val,
    Y_val
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size
)

# Inspect one batch
src_batch, dec_batch, target_batch = next(iter(train_loader))

print("Source batch:", src_batch.shape)
print("Decoder batch:", dec_batch.shape)
print("Target batch:", target_batch.shape)


Source batch: torch.Size([128, 8])
Decoder batch: torch.Size([128, 8])
Target batch: torch.Size([128, 8])


In [14]:

index = 0

source = X_train[index].tolist()
decoder_input = D_train[index].tolist()
target = Y_train[index].tolist()

print("SOURCE:        ", decode(source))
print("DECODER INPUT: ", [
    itos[i] for i in decoder_input
])
print("EXPECTED:      ", decode(target))

assert decode(target) == decode(source)[::-1]

print("\nDataset verification successful!")


SOURCE:         faffeaed
DECODER INPUT:  ['<BOS>', 'd', 'e', 'a', 'e', 'f', 'f', 'a']
EXPECTED:       deaeffaf

Dataset verification successful!


In [15]:
import math
import torch
import torch.nn as nn

# Vocabulary
vocab = list("abcdef") + ["<BOS>"]

stoi = {token: i for i, token in enumerate(vocab)}
itos = {i: token for token, i in stoi.items()}

VOCAB_SIZE = len(vocab)
D_MODEL = 64
MAX_LEN = 8

# Trainable embedding lookup table
token_embedding = nn.Embedding(
    num_embeddings=VOCAB_SIZE,
    embedding_dim=D_MODEL
)

print(token_embedding.weight.shape)

torch.Size([7, 64])


In [16]:
sample = torch.tensor([[0, 1, 2, 3, 4, 5]])

embedded = token_embedding(sample)

print("Input shape:", sample.shape)
print("Embedding shape:", embedded.shape)

Input shape: torch.Size([1, 6])
Embedding shape: torch.Size([1, 6, 64])


In [17]:
class SinusoidalPositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len=100):

        super().__init__()

        # Matrix containing positional encodings
        pe = torch.zeros(max_len, d_model)

        # Positions: 0, 1, 2, ..., max_len-1
        position = torch.arange(
            0, max_len, dtype=torch.float
        ).unsqueeze(1)

        # Frequency term for each sine/cosine pair
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-math.log(10000.0) / d_model)
        )

        # Even dimensions: sine
        pe[:, 0::2] = torch.sin(position * div_term)

        # Odd dimensions: cosine
        pe[:, 1::2] = torch.cos(position * div_term)

        # Add batch dimension: (1, max_len, d_model)
        pe = pe.unsqueeze(0)

        # Register as a buffer, not a trainable parameter
        self.register_buffer("pe", pe)

    def forward(self, x):

        seq_len = x.size(1)

        if seq_len > self.pe.size(1):
            raise ValueError("Sequence exceeds maximum positional encoding length")

        return x + self.pe[:, :seq_len, :].to(
            device=x.device, dtype=x.dtype
        )

In [19]:
class TokenPositionEmbedding(nn.Module):

    def __init__(self, vocab_size, d_model, max_len):

        super().__init__()

        self.d_model = d_model

        self.token_embedding = nn.Embedding(
            vocab_size, d_model
        )

        self.position_encoding = SinusoidalPositionalEncoding(
            d_model, max_len
        )

    def forward(self, token_ids):

        # 1. Convert token IDs into vectors
        x = self.token_embedding(token_ids)

        # 2. Scale embeddings
        x = x * math.sqrt(self.d_model)

        # 3. Add positional information
        x = self.position_encoding(x)

        return x

In [20]:
embedding_layer = TokenPositionEmbedding(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    max_len=MAX_LEN
)

In [21]:
source = torch.tensor(
    [[stoi[c] for c in "abcdefab"]],
    dtype=torch.long
)

decoder_input = torch.tensor(
    [[stoi["<BOS>"]] + [stoi[c] for c in "abcdefa"]],
    dtype=torch.long
)

source_vectors = embedding_layer(source)
decoder_vectors = embedding_layer(decoder_input)

print("Source IDs:", source.shape)
print("Source representations:", source_vectors.shape)

print("Decoder IDs:", decoder_input.shape)
print("Decoder representations:", decoder_vectors.shape)

Source IDs: torch.Size([1, 8])
Source representations: torch.Size([1, 8, 64])
Decoder IDs: torch.Size([1, 8])
Decoder representations: torch.Size([1, 8, 64])


In [22]:
class ScaledDotProductAttention(nn.Module):

    def __init__(self, dropout=0.0):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

    def forward(self, Q, K, V, mask=None):

        d_k = Q.size(-1)

        # 1. Calculate compatibility scores
        scores = torch.matmul(Q, K.transpose(-2, -1))

        # 2. Scale the scores
        scores = scores / math.sqrt(d_k)

        # 3. Apply mask, if supplied
        if mask is not None:
            scores = scores.masked_fill(mask, float("-inf"))

        # 4. Convert scores into probabilities
        attention_weights = torch.softmax(scores, dim=-1)

        # 5. Apply dropout to attention probabilities
        attention_weights = self.dropout(attention_weights)

        # 6. Weighted sum of Value vectors
        output = torch.matmul(attention_weights, V)

        return output, attention_weights

In [23]:
class MultiHeadAttention(nn.Module):

    def __init__(self, d_model, num_heads, dropout=0.0):

        super().__init__()

        assert d_model % num_heads == 0, \
            "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # Independent learned projections for Q, K and V
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)

        # Output projection W^O
        self.W_O = nn.Linear(d_model, d_model, bias=False)

        self.attention = ScaledDotProductAttention(dropout)

    def split_heads(self, x):

        batch_size, seq_len, _ = x.shape

        # (B, L, 64) -> (B, L, 4, 16)
        x = x.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.d_k
        )

        # (B, L, 4, 16) -> (B, 4, L, 16)
        return x.transpose(1, 2)

    def forward(self, query, key, value, mask=None):

        # 1. Linear projections
        Q = self.W_Q(query)
        K = self.W_K(key)
        V = self.W_V(value)

        # 2. Divide into heads
        Q = self.split_heads(Q)
        K = self.split_heads(K)
        V = self.split_heads(V)

        # 3. Attention independently in every head
        x, attention_weights = self.attention(
            Q, K, V, mask
        )

        # 4. Combine the heads
        x = x.transpose(1, 2).contiguous()

        batch_size, seq_len, _, _ = x.shape

        x = x.view(
            batch_size,
            seq_len,
            self.d_model
        )

        # 5. Final learned projection
        output = self.W_O(x)

        return output, attention_weights

In [24]:
D_MODEL = 64
NUM_HEADS = 4

mha = MultiHeadAttention(
    d_model=D_MODEL,
    num_heads=NUM_HEADS
)

# Use the embeddings from our previous session
x = embedding_layer(source)

output, weights = mha(x, x, x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)
print("Attention weights shape:", weights.shape)

Input shape: torch.Size([1, 8, 64])
Output shape: torch.Size([1, 8, 64])
Attention weights shape: torch.Size([1, 4, 8, 8])


In [26]:
# Select batch 0, head 0
head_0 = weights[0, 0]

print(head_0)
print("Row sums:", head_0.sum(dim=-1))

tensor([[1.7839e-13, 7.0753e-17, 9.9978e-01, 1.7026e-16, 2.1584e-04, 9.8457e-13,
         2.0034e-12, 5.7106e-16],
        [8.5966e-20, 6.1660e-01, 5.7381e-21, 1.5498e-11, 9.1289e-14, 9.4450e-10,
         5.1822e-20, 3.8340e-01],
        [1.4784e-09, 1.2492e-13, 3.8573e-28, 3.0848e-13, 1.0000e+00, 2.9536e-09,
         2.1651e-09, 6.6199e-14],
        [3.6196e-03, 1.3034e-07, 3.3811e-07, 2.2085e-22, 3.3064e-20, 9.8268e-01,
         1.3703e-02, 7.0604e-07],
        [3.6000e-07, 6.7117e-12, 9.9999e-01, 5.2149e-06, 5.3084e-14, 4.1672e-16,
         1.5143e-07, 2.8068e-12],
        [1.6242e-11, 3.7028e-06, 1.8666e-06, 9.9999e-01, 8.8356e-17, 6.7473e-12,
         2.3951e-11, 6.2884e-06],
        [1.0906e-14, 1.6436e-17, 9.9983e-01, 8.5753e-16, 1.7278e-04, 1.8393e-13,
         1.2977e-13, 1.3289e-16],
        [6.0121e-20, 6.1474e-01, 1.7133e-20, 5.7251e-11, 7.3515e-14, 3.3472e-10,
         3.6939e-20, 3.8526e-01]], grad_fn=<SelectBackward0>)
Row sums: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.

In [27]:
assert torch.allclose(
    head_0.sum(dim=-1),
    torch.ones(8),
    atol=1e-6
)

print("Attention probability check passed.")

Attention probability check passed.
